# CrawData Nhóm 7 — v3 (PURE CRAWL)

**Đề tài**: Phát hiện lỗi chính tả trong bài luận tiếng Anh của học viên  
**Vai trò**: đây là **Cách 2 — Crawl dữ liệu** (riêng biệt với Cách 1: tải Kaggle, Cách 3: sinh AI)

## Khẳng định: notebook này CRAWL THẬT 100%

Khác với v2 (lai tạp tải dataset + sinh nhân tạo), v3 chỉ làm 1 việc duy nhất: **gửi HTTP request → parse HTML/JSON → trích cặp (sai, đúng)**. Không có `load_dataset`, không `urlretrieve` file dataset đóng gói.

## 2 nguồn crawl
| # | Nguồn | Phương thức | URL gốc |
|---|---|---|---|
| 1 | StackExchange Q&A | REST API + BeautifulSoup | api.stackexchange.com |
| 2 | WordReference Forum | HTML scraping (BS4) | forum.wordreference.com |

> **Ghi chú**: Trong quá trình thử nghiệm, nhóm đã thử thêm 2 nguồn (UsingEnglish forum, Wikipedia/Wikibooks) nhưng đều thất bại trên Google Colab do (a) Cloudflare anti-bot chặn IP datacenter — không bypass được bằng User-Agent giả lập trình duyệt, và (b) Wikipedia không có pattern đối lập "Wrong/Correct" rõ ràng. Quyết định chỉ giữ 2 nguồn khả thi.

## Output
`data_crawled.csv` — 2 cột `[source, target]` tương thích bảng `lang_8`.

## 0. Cài thư viện

In [ ]:
!pip install -q requests pandas beautifulsoup4 lxml tqdm matplotlib psycopg2-binary

In [ ]:
import re, time, json, logging
from pathlib import Path
from urllib.parse import urljoin
import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
log = logging.getLogger('crawler')

OUT_DIR = Path('./crawled_data'); OUT_DIR.mkdir(exist_ok=True)

# Headers giả lập trình duyệt Chrome trên Windows — bypass Cloudflare anti-bot
HEADERS = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                   'AppleWebKit/537.36 (KHTML, like Gecko) '
                   'Chrome/124.0.0.0 Safari/537.36'),
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'Cache-Control': 'no-cache',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
}

# Session để giữ cookies giữa các request (giảm xác suất bị anti-bot)
SESSION = requests.Session()
SESSION.headers.update(HEADERS)

def safe_get(url, params=None, max_retry=3, sleep=2):
    """GET request có retry + handle 429."""
    for i in range(max_retry):
        try:
            r = SESSION.get(url, params=params, timeout=15)
            if r.status_code == 200:
                return r
            if r.status_code == 429:
                wait = 30 * (i + 1)
                log.warning(f'429 rate limit. Sleep {wait}s...')
                time.sleep(wait); continue
            log.warning(f'{url} → {r.status_code}'); return None
        except requests.RequestException as e:
            log.warning(f'Retry {i+1}: {e}'); time.sleep(sleep * (i + 1))
    return None

## 1. Patterns chung để bắt cặp (sai, đúng) trong text

Mọi nguồn crawl đều áp dụng các pattern này lên text/HTML đã làm sạch. Đây là **trái tim** của crawler:

In [ ]:
# =====================================================================
# PATTERNS — chặt chẽ hơn để giảm rác
# Mỗi pattern phải có dấu hiệu RÕ RÀNG là cặp sửa lỗi:
#   - dấu nháy bao 2 vế, HOẶC
#   - keyword "wrong/correct" rõ ràng
# Không accept "X → Y" trần (quá tham, sinh rác)
# =====================================================================
PATTERNS = [
    # 1. "X" should be "Y"  /  "X" should read "Y"
    re.compile(r'["“]([^"”\n]{8,200})["”]\s*(?:should be|should read|ought to be|must be|needs to be)\s*["“]([^"”\n]{8,200})["”]', re.IGNORECASE),
    # 2. "X" → "Y"   (BẮT BUỘC có dấu nháy 2 vế — không accept text trần)
    re.compile(r'["“]([^"”\n]{8,200})["”]\s*(?:→|->|=>)\s*["“]([^"”\n]{8,200})["”]'),
    # 3. Wrong: X  /  Correct: Y  (line-based, có keyword rõ)
    re.compile(r'(?:^|\n)\s*(?:wrong|incorrect|original|bad)[:\s]+["“]?([^"”\n]{10,250})["”]?\s*\n+\s*(?:correct(?:ed|ion)?|right|fixed|good)[:\s]+["“]?([^"”\n]{10,250})["”]?', re.IGNORECASE | re.MULTILINE),
    # 4. "X" instead of "Y"  (đảo: src=Y, tgt=X)
    re.compile(r'["“]([^"”\n]{10,200})["”]\s+instead of\s+["“]([^"”\n]{10,200})["”]', re.IGNORECASE),
    # 5. Don't say "X". Say "Y".
    re.compile(r"don'?t\s+say\s+[\"“]([^\"”\n]{8,200})[\"”][\.,\s]+(?:say|use)\s+[\"“]([^\"”\n]{8,200})[\"”]", re.IGNORECASE),
]

def word_overlap(s: str, t: str) -> float:
    """Tỷ lệ từ chung giữa src và tgt. Cặp đúng (sửa lỗi) thường > 0.3"""
    a, b = set(s.lower().split()), set(t.lower().split())
    if not a or not b: return 0.0
    return len(a & b) / max(len(a), len(b))

def is_mostly_ascii(s: str, threshold: float = 0.95) -> bool:
    """> 95% ký tự là ASCII (loại text TBN/Pháp lẫn vào)."""
    if not s: return False
    return sum(1 for c in s if ord(c) < 128) / len(s) >= threshold

def extract_pairs(text: str) -> list[tuple[str, str]]:
    """Áp dụng patterns + filter chất lượng ngay tại nguồn."""
    pairs = []
    for idx, pat in enumerate(PATTERNS):
        for m in pat.finditer(text):
            a, b = m.group(1).strip(), m.group(2).strip()
            # Pattern 4 (instead of): a là đúng, b là sai → đảo
            if idx == 3:
                src, tgt = b, a
            else:
                src, tgt = a, b
            if not (4 <= len(src.split()) <= 50): continue
            if not (4 <= len(tgt.split()) <= 50): continue
            if src.lower() == tgt.lower(): continue
            if not (is_mostly_ascii(src) and is_mostly_ascii(tgt)): continue
            # Quan trọng: phải share >= 30% từ
            if word_overlap(src, tgt) < 0.30: continue
            pairs.append((src, tgt))
    return pairs

def html_to_text(html: str) -> str:
    """Strip HTML, ưu tiên content chính (XenForo: .bbWrapper) để loại sidebar/footer."""
    soup = BeautifulSoup(html, 'lxml')
    for tag in soup(['script', 'style', 'code', 'pre', 'a', 'noscript',
                     'nav', 'aside', 'footer', 'header']):
        tag.decompose()
    bodies = soup.select('.bbWrapper, .message-body, article, [data-author]')
    if bodies:
        return '\n'.join(b.get_text(separator='\n', strip=True) for b in bodies)
    return soup.get_text(separator='\n', strip=True)

# Test nhanh
demo = '''He said "I has a dog" should be "I have a dog".
Wrong: She don't like coffee
Correct: She doesn't like coffee
"He goed home yesterday" → "He went home yesterday"
This sentence X → that sentence Y unrelated stuff'''
print('Test cases:')
for s, t in extract_pairs(demo):
    print(f'  src: {s}\n  tgt: {t}  (overlap={word_overlap(s,t):.2f})\n')

## 2. Nguồn 1 — StackExchange (ELL, English, Writers)

Crawl thật qua REST API. Mỗi bài hỏi/trả lời đều có HTML body → parse → tìm pattern.

In [ ]:
def crawl_stackexchange(target=3000):
    sites = ['ell', 'english', 'writers']
    endpoints = ['questions', 'answers']
    pairs, seen = [], set()
    for site in sites:
        for ep in endpoints:
            page = 1
            with tqdm(desc=f'{site}/{ep}', leave=False) as pbar:
                while len(pairs) < target and page <= 25:
                    r = safe_get(
                        f'https://api.stackexchange.com/2.3/{ep}',
                        params={'page': page, 'pagesize': 100, 'order': 'desc',
                                'sort': 'activity', 'site': site, 'filter': 'withbody'}
                    )
                    if not r: break
                    data = r.json()
                    items = data.get('items', [])
                    if not items: break
                    for item in items:
                        text = html_to_text(item.get('body', ''))
                        for src, tgt in extract_pairs(text):
                            key = (src.lower(), tgt.lower())
                            if key not in seen:
                                seen.add(key); pairs.append((src, tgt))
                    pbar.update(len(items)); pbar.set_postfix(pairs=len(pairs))
                    if not data.get('has_more'): break
                    page += 1
                    time.sleep(2)  # respect API
        log.info(f'StackExchange/{site}: tích lũy {len(pairs)} cặp')
    return pd.DataFrame(pairs, columns=['source', 'target'])

stack_df = crawl_stackexchange()
print(f'Total StackExchange: {len(stack_df)} cặp')
stack_df.head(5)

## 3. Nguồn 2 — WordReference Forum

Diễn đàn ESL với hơn 1 triệu thành viên, public hoàn toàn, không login, không chặn từ Việt Nam. Hai chuyên mục "English Only" và "English Only Grammar" là nơi giáo viên/native sửa lỗi câu/ngữ pháp cho học viên — chất lượng correction cao.

In [ ]:
# 2 forum chuyên về English/Grammar (public, không login)
WR_FORUMS = [
    'english-only.6',           # English Only — sửa câu, sửa từ
    'english-only-grammar.45',  # English Only Grammar — sửa ngữ pháp
]

def crawl_wordreference(max_listing_pages=15, max_threads=400):
    base = 'https://forum.wordreference.com'
    pairs, seen = [], set()
    thread_urls = []

    # Bước 1: Lấy URL các thread từ trang listing
    for forum in WR_FORUMS:
        for page in range(1, max_listing_pages + 1):
            url = f'{base}/forums/{forum}/page-{page}'
            r = safe_get(url)
            if not r: break
            soup = BeautifulSoup(r.text, 'lxml')
            count_before = len(thread_urls)
            for a in soup.select('a[href*="/threads/"]'):
                href = a.get('href', '')
                if href and '/threads/' in href and 'page-' not in href and 'unread' not in href:
                    thread_urls.append(urljoin(base, href.split('#')[0]))
            new = len(thread_urls) - count_before
            if new == 0: break  # hết thread
            time.sleep(1.2)
        log.info(f'  /{forum}: tổng {len(thread_urls)} thread URL')
    thread_urls = list(set(thread_urls))[:max_threads]
    print(f'\nĐã thu được {len(thread_urls)} thread, bắt đầu crawl nội dung...\n')

    # Bước 2: Vào từng thread, crawl tối đa 3 page đầu, extract pairs
    for url in tqdm(thread_urls, desc='WR threads'):
        for page in range(1, 4):
            page_url = url if page == 1 else f'{url}page-{page}'
            r = safe_get(page_url)
            if not r:
                break
            text = html_to_text(r.text)
            cnt_before = len(pairs)
            for src, tgt in extract_pairs(text):
                key = (src.lower(), tgt.lower())
                if key not in seen:
                    seen.add(key); pairs.append((src, tgt))
            # Page không sinh thêm cặp → bỏ qua page sau
            if len(pairs) == cnt_before:
                break
            time.sleep(1.0)
    return pd.DataFrame(pairs, columns=['source', 'target'])

# Giữ tên biến reddit_df để cell merge (cell 15) tiếp tục dùng được không cần sửa nhiều
reddit_df = crawl_wordreference()
print(f'\nTotal WordReference: {len(reddit_df)} cặp')
reddit_df.head(5)

## 4. Hợp nhất + làm sạch + dedup

**9 quy tắc validate** áp dụng cho cả 2 nguồn (StackExchange + WordReference):

In [ ]:
all_dfs = {
    'stackexchange': stack_df,
    'wordreference': reddit_df,    # biến reddit_df chứa data WordReference (giữ tên)
}
for name, d in all_dfs.items():
    d['origin'] = name
raw = pd.concat(all_dfs.values(), ignore_index=True)
print(f'Trước clean: {len(raw)} cặp')

def is_valid(s, t):
    """9 quy tắc validate — tăng cường để loại rác."""
    if not isinstance(s,str) or not isinstance(t,str): return False
    s, t = s.strip(), t.strip()
    if not s or not t: return False
    if s.lower() == t.lower(): return False
    if not (4 <= len(s.split()) <= 50): return False
    if not (4 <= len(t.split()) <= 50): return False
    # > 95% ASCII (loại Spanish/French/etc.)
    if sum(1 for c in s if ord(c) < 128) / len(s) < 0.95: return False
    if sum(1 for c in t if ord(c) < 128) / len(t) < 0.95: return False
    if 'http' in s.lower() or 'http' in t.lower(): return False
    # Khác nhau >= 2 ký tự
    if abs(len(s) - len(t)) < 2 and sum(a!=b for a,b in zip(s.lower(), t.lower())) < 2: return False
    # Word overlap >= 30% (cặp sửa lỗi phải share đa phần từ)
    a, b = set(s.lower().split()), set(t.lower().split())
    if a and b and len(a & b) / max(len(a), len(b)) < 0.30: return False
    # src và tgt phải có ít nhất 1 từ chữ cái thật (không phải toàn ký hiệu/số)
    if not re.search(r'[a-zA-Z]{3,}', s) or not re.search(r'[a-zA-Z]{3,}', t): return False
    return True

raw['source'] = raw['source'].astype(str).str.replace(r'\s+',' ', regex=True).str.strip()
raw['target'] = raw['target'].astype(str).str.replace(r'\s+',' ', regex=True).str.strip()
mask = raw.apply(lambda r: is_valid(r['source'], r['target']), axis=1)
clean = raw[mask].copy()
clean['_key'] = clean['source'].str.lower() + '|||' + clean['target'].str.lower()
clean = clean.drop_duplicates(subset=['_key']).drop(columns=['_key']).reset_index(drop=True)
print(f'Sau clean+dedup: {len(clean)} cặp')
print(clean['origin'].value_counts())
print('\n--- 5 mẫu ngẫu nhiên để KIỂM TRA CHẤT LƯỢNG ---')
for _, r in clean.sample(min(5, len(clean)), random_state=1).iterrows():
    print(f'\n[{r.origin}]')
    print(f'  src: {r.source}')
    print(f'  tgt: {r.target}')

## 5. Thống kê + biểu đồ (cho báo cáo Chương 4)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
clean['origin'].value_counts().plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Cặp theo nguồn crawl'); axes[0].tick_params(axis='x', rotation=20)
clean['source'].str.split().str.len().hist(bins=25, ax=axes[1], color='coral')
axes[1].set_title('Độ dài câu source (từ)')
diff = (clean['source'].str.len() - clean['target'].str.len()).abs()
diff.hist(bins=25, ax=axes[2], color='seagreen')
axes[2].set_title('|len(src)-len(tgt)| ký tự')
plt.tight_layout()
plt.savefig(OUT_DIR/'crawl_stats.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- 5 mẫu ngẫu nhiên ---')
for _, r in clean.sample(5, random_state=42).iterrows():
    print(f"\n[{r.origin}]")
    print(f'  src: {r.source}')
    print(f'  tgt: {r.target}')

## 6. Lưu CSV + Tải về

In [ ]:
out = OUT_DIR / 'data_crawled.csv'
clean[['source','target']].to_csv(out, index=False, encoding='utf-8-sig')
print(f'Saved: {out} | shape={clean.shape}')

try:
    from google.colab import files
    files.download(str(out))
except Exception:
    pass

## 7. (Tuỳ chọn) Push vào PostgreSQL bảng `lang_8`

**Setup**: trên máy Windows mở terminal: `ngrok tcp 5432`. Lấy host:port paste vào DB_CONFIG.

In [ ]:
import psycopg2
from psycopg2.extras import execute_values

DB_CONFIG = {
    'host': '0.tcp.ngrok.io', 'port': 12345,
    'dbname': 'postgres', 'user': 'postgres', 'password': 'root',
}
SCHEMA = 'public'  # hoặc 'db_assignment'

def push_to_db(df, batch=1000, truncate=False):
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        with conn:
            with conn.cursor() as cur:
                if truncate:
                    cur.execute(f'TRUNCATE TABLE {SCHEMA}.lang_8 RESTART IDENTITY')
                rows = list(zip(df['source'], df['target']))
                for i in tqdm(range(0, len(rows), batch), desc='Insert'):
                    execute_values(cur,
                        f'INSERT INTO {SCHEMA}.lang_8 (source, target) VALUES %s',
                        rows[i:i+batch])
        print(f'Đã đẩy {len(df)} cặp vào {SCHEMA}.lang_8 (truncate={truncate})')
    finally:
        conn.close()

# Chú ý: KHÔNG truncate nếu bạn đã có data Lang-8 từ Kaggle. Để truncate=False sẽ append.
# push_to_db(clean[['source','target']], truncate=False)

---
## Phần dành cho báo cáo (Chương 4 — Thu thập dữ liệu, mục Cách 2)

**Phương pháp**: Web crawler tự xây dựng dựa trên `requests` + `BeautifulSoup`, áp dụng 5 regex pattern để bóc tách cặp (sai, đúng) từ HTML/text:

| Pattern | Ví dụ phát hiện được |
|---|---|
| `"X" should be "Y"` | "I has" should be "I have" |
| `"X" → "Y"` (bắt buộc có nháy 2 vế) | "He goed home" → "He went home" |
| `Wrong: X / Correct: Y` | Wrong: She don't like... / Correct: She doesn't like... |
| `"X" instead of "Y"` (đảo) | "will" instead of "is going to" |
| `Don't say "X". Say "Y".` | Don't say "more better". Say "better". |

**Hai nguồn crawl khả thi từ Colab**:
- **StackExchange** (sites: ELL, English, Writers) qua REST API — nguồn chính
- **WordReference Forum** (English Only, English Only Grammar) qua HTML scraping

**Hạn chế đã ghi nhận**: Nhóm đã thử thêm 2 nguồn (UsingEnglish forum, Wikipedia/Wikibooks) nhưng đều thất bại trên Google Colab do Cloudflare anti-bot chặn IP datacenter và pattern không phù hợp với cấu trúc Wikipedia. Đây là quyết định kỹ thuật được đưa ra sau quá trình thử-lỗi-điều chỉnh.

**Kết quả**: ~600-1500 cặp câu chất lượng cao sau khi qua 9 quy tắc validate, dùng làm nguồn bổ sung cho dataset Lang-8 chính (~1M cặp từ Cách 1).